This model uses a Convolutional Neural Network design in order to classify images, using various convolutions that are developed and refined overtime across several training sessions in order to properly classify images into various categories.

The programming will be done in two ways. First, pytorch will be used in order to build a simple CNN. Then, it'll be done from scratch using numpy as the mathematics library.

Training data will be stored in a folder named "Datasets" and in two folders - "Cat" and "Dog". The dataset was obtained from here: https://www.kaggle.com/datasets/bhavikjikadara/dog-and-cat-classification-dataset?resource=download.

Test data will be stored within "Datasets" in "\_TEST\_", with the same format as is used for the training data. The dataset was obtained from here:
https://www.kaggle.com/datasets/abdullahbakr7/cats-vs-dogs-image-classification/data

In [44]:
# Initial Imports.
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

import numpy as np
import os

In [45]:
# Setting torch to use GPU acceleration if possible.
device = torch.device("cpu")

if torch.cuda.is_available():
    device = torch.device("cuda")

torch.set_default_device(device)
print(f"Using device: {torch.get_default_device()}")

Using device: cpu


**Pytorch Model**

In [41]:
# Preparing dataset for Pytorch
class PytorchDataset(Dataset):
    def __init__(self, root = "Datasets"):
        self.transform = transforms.Compose({
            transforms.Resize((32, 32)),
            transforms.ToTensor()
        })

        self.images = []
        self.labels = []

        self.classes = {
            "Cat": 0,
            "Dog": 1
        }

        # Load dataset
        for className, label in self.classes.items():
            classPath = os.path.join(root, className)

            for imageName in os.listdir(classPath):
                imagePath = os.path.join(classPath, imageName)
                self.images.append(imagePath)
                self.labels.append(label)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        imagePath = self.images[idx]

        # Load and prepare image.
        image = Image.open(imagePath).convert("RGB")
        image = self.transform(image)

        label = self.labels[idx]

        return image, label

In [42]:
# Generate the dataset. This dataset can then be used in as many models as we want.
dataset = PytorchDataset()

In [699]:
class PytorchModel(nn.Module):
    def __init__(self):
        super(PytorchModel, self).__init__()
        self.dataset = dataset
        
        self.conv1 = nn.Conv2d(3, 6, 5)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(6, 16, 5)
        self.l1 = nn.Linear(16 * 5 * 5, 120)
        self.l2 = nn.Linear(120, 84)
        self.l3 = nn.Linear(84, 2)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(-1, 16 * 5 * 5)
        x = F.relu(self.l1(x))
        x = F.relu(self.l2(x))
        return self.l3(x)

In [49]:
# Prepare Training.
model = PytorchModel()

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr = 0.001, momentum = 0.9)

trainLoader = DataLoader(dataset, batch_size = 32, shuffle = True)

In [50]:
# Train the model.
epochs = 50

for epoch in range(epochs):
    runningLoss = 0.0

    for i, data in enumerate(trainLoader):
        inputs, labels = data
        optimizer.zero_grad()

        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        runningLoss += loss.item()
        
    print("Epoch #" + str(epoch + 1) + ": " + str(runningLoss / (i + 1)))

Epoch #1: 0.8569628706063761
Epoch #2: 0.6972201862146178
Epoch #3: 0.6916871057142078
Epoch #4: 0.6879117887495728
Epoch #5: 0.6834235925351262
Epoch #6: 0.676551914428506
Epoch #7: 0.6719291611858036
Epoch #8: 0.6694031382155845
Epoch #9: 0.6654498059578868
Epoch #10: 0.6616954214280218
Epoch #11: 0.6584751423057693
Epoch #12: 0.6528989552232005
Epoch #13: 0.6486266833132185
Epoch #14: 0.6452107779738848
Epoch #15: 0.6386637788294526
Epoch #16: 0.6351612894736287
Epoch #17: 0.6291092509961189
Epoch #18: 0.6253044179181004
Epoch #19: 0.6178557050731176
Epoch #20: 0.6121788419344846
Epoch #21: 0.6074909523243794
Epoch #22: 0.5956708235890055
Epoch #23: 0.587616338944801
Epoch #24: 0.579742373693782
Epoch #25: 0.5726036923696928
Epoch #26: 0.560346063140713
Epoch #27: 0.5535015456206963
Epoch #28: 0.5479954588596169
Epoch #29: 0.5394586218744898
Epoch #30: 0.5330769811825984
Epoch #31: 0.5287493561844692
Epoch #32: 0.5204749433966853
Epoch #33: 0.5128717956030765
Epoch #34: 0.5068889599

In [46]:
# Test the model.
testDataset = PytorchDataset(os.path.join("Datasets", "_TEST_"))
testLoader = DataLoader(testDataset, batch_size = 4, shuffle = False)

correct = 0
total = 0

with torch.no_grad():
    for data in testLoader:
        images, labels = data
        outputs = model(images)

        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print("Accuracy: " + str(100 * correct / total))

Accuracy: 59.8


**Numpy Model**

In [46]:
# Initial imports.
import math
import random

In [48]:
# Defining layers that we'll use throughout the mode. This includes Conv2D, Pooling, ReLu, and Linear.
class Layer:
    def __init__(self):
        self.parameters = {}

    def setup(self):
        self.gradients = {}
        for parameter in self.parameters.keys():
            self.gradients[parameter] = np.zeros_like(self.parameters[parameter])

    def forward(self, x):
        return x

    def backward(self, lossGradient):
        return lossGradient

    def adjust_parameters(self, learningRate, momentum = 0.0):
        for parameter in self.parameters.keys():
            self.parameters[parameter] -= self.gradients[parameter] * learningRate

        self.reset_gradients(momentum)

    def reset_gradients(self, momentum = 0.0):
        for parameter in self.gradients.keys():
            self.gradients[parameter] *= momentum

In [49]:
class Linear(Layer):
    def __init__(self, inputSize, outputSize):
        self.parameters = {
            "Weights": np.random.randn(inputSize, outputSize) * np.sqrt(2 / inputSize),
            "Bias": np.zeros((1, outputSize))
        }

        self.prevInput = None
        
        self.setup()

    def forward(self, x):
        self.prevInput = x
        return np.matmul(x, self.parameters["Weights"]) + self.parameters["Bias"]

    def backward(self, lossGradient):
        weightGradient = np.repeat(np.transpose(self.prevInput), self.parameters["Weights"].shape[1], axis = 1)
        self.gradients["Weights"] += weightGradient * np.repeat(lossGradient, self.parameters["Weights"].shape[0], axis = 0)
        self.gradients["Bias"] += lossGradient

        return np.matmul(lossGradient, np.transpose(self.parameters["Weights"]))

In [50]:
class Conv2D(Layer):
    def __init__(self, inputSize, outputSize, kernel, stride = 1):
        kernelSize = kernel[0] * kernel[1] * inputSize
        
        self.kernel = kernel
        self.stride = stride

        self.parameters = {
            "Filter": np.random.randn(*kernel, inputSize, outputSize) * np.sqrt(2 / kernelSize),
            "Bias": np.zeros(outputSize)
        }

        self.prevInput = None
        self.setup()

    def forward(self, x):
        self.prevInput = x

        rowCount = ((x.shape[0] - self.kernel[0]) // self.stride) + 1
        colCount = ((x.shape[1] - self.kernel[1]) // self.stride) + 1
        
        output = np.zeros((
            rowCount,
            colCount,
            self.parameters["Filter"].shape[3]
        ))

        for row in range(rowCount):
            for col in range(colCount):
                inputRow = row * self.stride
                inputCol = col * self.stride

                subMatrix = x[inputRow:inputRow + self.kernel[0], inputCol:inputCol + self.kernel[1], :]

                for channel in range(self.parameters["Filter"].shape[3]):
                    output[row, col, channel] = np.sum(subMatrix * self.parameters["Filter"][:, :, :, channel]) + self.parameters["Bias"][channel]

        return output

    def backward(self, lossGradient):
        filterGradient = np.zeros_like(self.parameters["Filter"])
        biasGradient = np.zeros_like(self.parameters["Bias"])

        inputGradient = np.zeros_like(self.prevInput)

        rowCount, colCount, outputCount = lossGradient.shape

        for row in range(rowCount):
            for col in range(colCount):
                inputRow = row * self.stride
                inputCol = col * self.stride

                subMatrix = self.prevInput[inputRow : inputRow + self.kernel[0], inputCol:inputCol + self.kernel[1], :]

                for channel in range(outputCount):
                    gradient = lossGradient[row, col, channel]

                    filterGradient[:, :, :, channel] += subMatrix * gradient
                    biasGradient[channel] += gradient

                    inputGradient[inputRow:inputRow + self.kernel[0], inputCol:inputCol + self.kernel[1], :] += self.parameters["Filter"][:, :, :, channel] * gradient

        self.gradients["Filter"] += filterGradient
        self.gradients["Bias"] += biasGradient

        return inputGradient

In [51]:
class LeakyReLU(Layer):
    def __init__(self):
        self.prevInput = None
        self.parameters = {}
        self.setup()

    def forward(self, x):
        self.prevInput = x
        return np.maximum(0.01 * x, x)

    def backward(self, lossGradient):
        output = np.where(self.prevInput <= 0, 0.01, 1)
        return output * lossGradient

In [52]:
class MaxPool(Layer):
    def __init__(self, kernel, stride = 1):
        self.kernel = kernel
        self.stride = stride
        
        self.prevInput = None
        self.parameters = {}
        self.setup()

    def forward(self, x):
        self.prevInput = x

        rowCount = ((x.shape[0] - self.kernel[0]) // self.stride) + 1
        colCount = ((x.shape[1] - self.kernel[1]) // self.stride) + 1
        
        output = np.zeros((
            rowCount,
            colCount,
            x.shape[2]
        ))

        for row in range(rowCount):
            for col in range(colCount):
                inputRow = row * self.stride
                inputCol = col * self.stride

                subMatrix = x[inputRow : inputRow + self.kernel[0], inputCol : inputCol + self.kernel[1], :]
                output[row, col, :] = np.max(subMatrix, axis = (0, 1))

        return output

    def backward(self, lossGradient):
        inputGradient = np.zeros_like(self.prevInput)
        
        rowCount, colCount, outputCount = lossGradient.shape
        channelIndices = np.arange(outputCount)

        for row in range(rowCount):
            for col in range(colCount):
                inputRow = row * self.stride
                inputCol = col * self.stride

                subMatrix = self.prevInput[inputRow : inputRow + self.kernel[0], inputCol : inputCol + self.kernel[1], :]
                flattened = subMatrix.reshape(self.kernel[0] * self.kernel[1], outputCount)
                maxIndices = np.argmax(flattened, axis = 0)

                maxRows = maxIndices // self.kernel[1]
                maxCols = maxIndices % self.kernel[1]

                inputGradient[inputRow + maxRows, inputCol + maxCols, channelIndices] += lossGradient[row, col, :]

        return inputGradient

In [53]:
# Dataset Class (will hold matricized versions of every image).
class Dataset:
    def __init__(self, imageDimensions = [32, 32]):
        self.imageDimensions = imageDimensions
        self.load_dataset()

    def load_dataset(self):
        # Should only be run once, as it takes a while.
        self.dataset = {
            "Cat": [],
            "Dog": []
        }

        for folder in self.dataset.keys():
            targetFolder = os.path.join("Datasets", folder)
            for file in os.listdir(targetFolder):
                imagePath = os.path.join(targetFolder, file)
                self.dataset[folder].append(self.convert_image(imagePath))

    def convert_image(self, imagePath):
        image = Image.open(imagePath)
        image = image.resize((self.imageDimensions[0], self.imageDimensions[1]))
        return np.array(image)

In [54]:
dataset = Dataset()

In [55]:
class Model:
    def __init__(self):
        self.conv1 = Conv2D(4, 6, [5, 5])
        self.relu1 = LeakyReLU()

        self.pool1 = MaxPool([2, 2], 2)

        self.conv2 = Conv2D(6, 16, [5, 5])
        self.relu2 = LeakyReLU()

        self.pool2 = MaxPool([2, 2], 2)

        self.l1 = Linear(16 * 5 * 5, 120)
        self.relu3 = LeakyReLU()

        self.l2 = Linear(120, 84)
        self.relu4 = LeakyReLU()

        self.l3 = Linear(84, 2)

    def forward(self, x):
        # Each input is between 0 and 255, so divide them by 255 to reduce them to between 0.0 and 1.0.
        reducedX = x / 255

        # If the image is greyscale, extend it to RGB.
        if reducedX.ndim < 3:
            reducedX = np.dstack((reducedX, reducedX, reducedX, np.ones_like(reducedX)))

        # If the image doesn't have an alpha channel either, add it.
        elif reducedX.shape[2] <= 3:
            reducedX = np.dstack((reducedX[:, :, 0], reducedX[:, :, 1], reducedX[:, :, 2], np.ones_like(reducedX[:, :, 0])))

        # Now pass it through our convolution layers.
        convX = self.pool2.forward(
            self.relu2.forward(
                self.conv2.forward(
                    self.pool1.forward(
                        self.relu1.forward(
                            self.conv1.forward(reducedX)
                        )
                    )
                )
            )
        )

        # Reshape for linear layers.
        linearX = convX.reshape(1, 16 * 5 * 5)

        # Finally, pass it through the linear layers.
        return self.l3.forward(
            self.relu4.forward(
                self.l2.forward(
                    self.relu3.forward(
                        self.l1.forward(linearX)
                    )
                )
            )
        )

    def backward(self, lossGradient):
        # Linear Backpropagation
        linearGradient = self.l1.backward(self.relu3.backward(self.l2.backward(self.relu4.backward(self.l3.backward(lossGradient)))))

        # Reshape back into what we need for convolutional networks.
        convGradients = linearGradient.reshape(5, 5, 16)
        
        self.conv1.backward(self.relu1.backward(self.pool1.backward(self.conv2.backward(self.relu2.backward(self.pool2.backward(convGradients))))))

    def adjust_parameters(self, learningRate, momentum = 0.0):
        self.conv1.adjust_parameters(learningRate, momentum)
        self.conv2.adjust_parameters(learningRate, momentum)
        self.l1.adjust_parameters(learningRate, momentum)
        self.l2.adjust_parameters(learningRate, momentum)
        self.l3.adjust_parameters(learningRate, momentum)

In [56]:
# Calculates and returns the loss through softmax cross-entropy.
def cross_entropy_loss(expected, result):
    # First, softmax the results.
    shifted = result - np.max(result, axis = 1, keepdims = True) # Prevents NaN
    exp = np.exp(shifted)
    probabilities = exp / np.sum(exp, axis = 1, keepdims = True)

    loss = -np.sum(expected * np.log(probabilities + 1e-12))
    lossGradient = probabilities - expected
    
    return loss, lossGradient, np.argmax(probabilities) == np.argmax(expected)

In [60]:
class Optimizer:
    def __init__(self, learningRateMin, learningRateMax, warmUpPercentage, linear = True):
        self.learningRateMin = learningRateMin
        self.learningRateMax = learningRateMax
        self.warmUpPercentage = warmUpPercentage
        self.linear = linear

    def calculate_learning_rate(self, epoch, totalEpochs):
        warmUpEpoch = totalEpochs * self.warmUpPercentage
        learningRateDiff = self.learningRateMax - self.learningRateMin

        if epoch <= warmUpEpoch: # Warm-up Stage.
            progress = epoch / warmUpEpoch
            
            if self.linear:
                return self.learningRateMin + (learningRateDiff * progress)

            else:
                return self.learningRateMin + (learningRateDiff * math.sin(progress * math.pi / 2))

        else:
            progress = (epoch - warmUpEpoch) / (totalEpochs - warmUpEpoch)

            if self.linear:
                return self.learningRateMax - (learningRateDiff * progress)

            else:
                return self.learningRateMax - (learningRateDiff * math.sin(progress * math.pi / 2))

In [70]:
# Train a model.
epochs = 100
batchSize = 32 # Final batch size is batchSize * classificationNumber
momentum = 0.9 # 0.0 means no momentum, 1.0 means non-degrading momentum.

optimizer = Optimizer(
    learningRateMin = 0.0001,
    learningRateMax = 0.001,
    warmUpPercentage = 0.1,
    linear = False
)

model = Model()

for epoch in range(1, epochs + 1):
    epochLearningRate = optimizer.calculate_learning_rate(epoch - 1, epochs)
    totalLoss = 0.0

    corrects = 0

    # Generate the batch.
    batch = []
    classifications = list(dataset.dataset.keys())

    for i, classification in enumerate(classifications):
        newBatch = random.sample(dataset.dataset[classification], batchSize)
        expected = np.zeros((1, len(classifications)))
        expected[0, i] = 1

        batch += [[element, expected] for element in newBatch]

    random.shuffle(batch)

    # Loop through the batch.
    for inputElement, expected in batch:
        result = model.forward(inputElement)
        loss, lossGradient, wasCorrect = cross_entropy_loss(expected, result)
        totalLoss += loss / len(batch) # Logging purposes only.
        model.backward(lossGradient)

        if wasCorrect:
            corrects += 1
    
    model.adjust_parameters(epochLearningRate, momentum)

    print(f"Epoch #{epoch}:\n     Loss: {totalLoss}\n     Accuracy - {corrects / len(batch)}\n")

Epoch #1:
     Loss: 0.9977710952679234
     Accuracy - 0.5

Epoch #2:
     Loss: 1.132185351492433
     Accuracy - 0.5

Epoch #3:
     Loss: 0.7327731804255562
     Accuracy - 0.5

Epoch #4:
     Loss: 0.9421016657139268
     Accuracy - 0.5

Epoch #5:
     Loss: 0.7024131622500858
     Accuracy - 0.484375

Epoch #6:
     Loss: 0.7188711079081218
     Accuracy - 0.5

Epoch #7:
     Loss: 0.696411902695004
     Accuracy - 0.46875

Epoch #8:
     Loss: 0.6901681826021236
     Accuracy - 0.546875

Epoch #9:
     Loss: 0.691165276026197
     Accuracy - 0.59375

Epoch #10:
     Loss: 0.6948668875817015
     Accuracy - 0.484375

Epoch #11:
     Loss: 0.6942093032251794
     Accuracy - 0.5

Epoch #12:
     Loss: 0.6934148319722454
     Accuracy - 0.5

Epoch #13:
     Loss: 0.6936242709256017
     Accuracy - 0.5

Epoch #14:
     Loss: 0.6947574937074076
     Accuracy - 0.5

Epoch #15:
     Loss: 0.6963121252959613
     Accuracy - 0.5

Epoch #16:
     Loss: 0.6926129695744773
     Accuracy - 0.

In [ ]:
# Load test dataset.
testImages = {
    "Cat": [],
    "Dog": []
}

for classification in testImages.keys():
    folderPath = os.path.join("Datasets", "_TEST_", classification)

    for image in os.listdir(folderPath):
        testImages[classification].append(os.path.join(folderPath, image))

In [ ]:
# Test the model.
corrects = {}

for classification in testImages.keys():
    corrects[classification] = 0

    for image in testImages[classification]:
        result = model.forward(dataset.convert_image(image))
        chosenClassification = list(dataset.dataset.keys())[np.argmax(result)]

        if chosenClassification == classification:
            corrects[classification] += 1

successPercentages = {}
total = 0
totalCorrects = 0

for classification in corrects.keys():
    successPercentages[classification] = (corrects[classification] / len(testImages[classification])) * 100
    total += len(testImages[classification])
    totalCorrects += corrects[classification]

print("Success Rate: " + str(100 * totalCorrects / total) + "%")

for classification in corrects.keys():
    print(classification + ": " + str(successPercentages[classification]) + "%")